In [ ]:
###
# セットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

LOCAL_DATA_DIR = Path("/content/data/cats_vs_dogs")

if not LOCAL_DATA_DIR.exists():
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    !unzip -q /content/drive/MyDrive/cnn-hands-on/data/cats_vs_dogs.zip -d /content/data/cats_vs_dogs


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# GPUが使える場合はGPUを、使えない場合はCPUを使用する設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")

In [ ]:
_FRAC_MAP = {"small":0.2, "medium":0.5, "large":1.0}

# ToDo:データセット実装
class CDDataset(Dataset):
  def __init__(self, df, data_dir, transform=None):
    self.df = df.reset_index(drop=True)
    self.data_dir = data_dir
    self.transform = transform
  
  def __len__(self):
    return len(self.df)
  
  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    image = Image.open(os.path.join(self.data_dir, row["filepath"])).convert("RGB")
    if self.transform:
      image = self.transform(image)
    return image, row["label"]

# ToDo:データローダ実装
def get_dc_dataloaders(data_dir, data_size="small", batch_size=32):
  frac = _FRAC_MAP.get(data_size)
  if frac is None:
    raise ValueError(
        "FRAC ERROR"
    )
  
  df = pd.read_csv(os.path.join(data_dir, "labels.csv"))
  transform = transforms.Compose(
      [transforms.Resize((128,128)), transforms.ToTensor(),transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
  )

  def make_loader(split, shuffle):
    dataset = CDDataset(
        df[df["split"] == split].sample(frac=frac, random_state=61),
        data_dir,
        transform,
    )
    kwargs = {"num_workers": 2, "pin_memory": True} if shuffle else {}
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, **kwargs)
  
  return (
      make_loader("train", True),
      make_loader("val", False),
      make_loader("test", False),
  )

In [ ]:
train_loader, val_loader, test_loader = get_dc_dataloaders(
    data_dir=LOCAL_DATA_DIR,
    data_size="large",
    batch_size=32
)

In [ ]:
# ToDo:ResNet18の実装
class BasicBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()
    self.conv1 = nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
    )
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.conv2 = nn.Conv2d(
        out_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False,
    )
    self.bn2 = nn.BatchNorm2d(out_channels)

    self.shortcut = nn.Sequential()
    if stride != 1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(
              in_channels,
              out_channels,
              kernel_size=1,
              stride=stride,
              bias=False,
          ),
          nn.BatchNorm2d(out_channels),
      )

  def forward(self, x):
    identity = self.shortcut(x)

    out = self.conv1(x)
    out = self.bn1(out)
    out = F.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    out = out + identity
    out = F.relu(out)
    return out


class ResNet18(nn.Module):
  def __init__(self, num_classes=2):
    super().__init__()
    self.in_channels = 64

    self.stem = nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    )

    self.layer1 = self._make_layer(out_channels=64, blocks=2, stride=1)
    self.layer2 = self._make_layer(out_channels=128, blocks=2, stride=2)
    self.layer3 = self._make_layer(out_channels=256, blocks=2, stride=2)
    self.layer4 = self._make_layer(out_channels=512, blocks=2, stride=2)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(512, num_classes)

  def _make_layer(self, out_channels, blocks, stride):
    layers = [BasicBlock(self.in_channels, out_channels, stride)]
    self.in_channels = out_channels

    for _ in range(1, blocks):
      layers.append(BasicBlock(out_channels, out_channels))

    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.stem(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)
    return x


model = ResNet18(num_classes=2).to(device)
print(model)


In [ ]:
# 学習の前準備

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 10
train_loss_list, val_loss_list, val_acc_list = [], [], []

best_val_loss = float("inf")

os.makedirs(ROOT_PATH / "models", exist_ok=True)
save_path = ROOT_PATH / "models" / "07_resnet18.pth"


In [ ]:
# 学習ループ実装

print("学習開始")
for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Train ", leave=False)
    # バッチごとにデータを取り出して学習
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        
        # ToDo:ステップの更新
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()

        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    # 1エポック分の平均Lossを記録
    epoch_train_loss = running_train_loss / len(train_loader)
    train_loss_list.append(epoch_train_loss)
    
    # --- Validation ---
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Val ", leave=False)
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_acc = 100 * correct / total

    val_loss_list.append(epoch_val_loss)
    val_acc_list.append(epoch_val_acc)

    # ToDo:モデルの保存
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), save_path)
        mark = "Best Model Saved"
    else:
        mark = ""

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | "
          f"Val Acc: {epoch_val_acc:.2f}% {mark}")
 
print("学習完了")

In [ ]:
###
# 5. 結果の描画 (グラフ)
###

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- Loss（誤差）の推移 ---
ax1.plot(train_loss_list, label='Train Loss', color='blue', marker='o')
ax1.plot(val_loss_list, label='Validation Loss', color='orange', marker='o')


ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# --- Accuracy（正答率）の推移 ---
ax2.plot(val_acc_list, label='Validation Accuracy', color='green', marker='o')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
###
# 6. テストデータで評価
###

model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

running_test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="Test", leave=False)
    for images, labels in test_bar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = running_test_loss / len(test_loader)
test_acc = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")